# ND2 Unified Discovery Notebook: Fase 6.5 - Descubrimiento de la Recurrencia Exacta

Objetivo: Encontrar la Fórmula de Recurrencia de Bonnet: 
P_L = ((2L-1)/L) * x * P_{L-1} - ((L-1)/L) * P_{L-2}

Target: P_L
Variables: l_order, x (cos_theta), p_prev1 (L-1), p_prev2 (L-2)

In [ ]:
# 0. Sincronización Robusta (Hard Reset)
import os
is_colab = os.path.exists('/content')
base_path = '/content' if is_colab else '/kaggle/working'
target_dir = os.path.join(base_path, 'ND2')
if not os.path.exists(target_dir): os.system(f"git clone https://github.com/manramirezpi/ND2.git {target_dir}")
os.chdir(target_dir)
os.system("git fetch origin")
os.system("git reset --hard origin/main")
print(f"\n[OK] Entorno de Recurrencia listo en: {os.getcwd()}")

In [ ]:
# 1. Setup
!pip install gplearn setproctitle geopandas
!mkdir -p weights Multipolos/data/nd2_format Multipolos/results
if not os.path.exists('weights/checkpoint.pth'):
    !wget -O weights/checkpoint.pth https://github.com/yuzhTHU/ND2/releases/download/checkpoint.pth/checkpoint.pth

In [ ]:
# 2. GENERAR DATASET DE RECURRENCIA INFORMADA
import numpy as np, json, os
from scipy.special import legendre

def generate_recurrence_data(num_samples_per_l=1000):
    # Empezamos en l=2 porque necesitamos l-1 y l-2
    all_l = [2, 3, 4, 5]
    
    data_node = {"l_order": [], "x": [], "p_prev1": [], "p_prev2": []}
    targets = []
    
    for l in all_l:
        theta = np.random.uniform(0, np.pi, num_samples_per_l)
        x_vals = np.cos(theta)
        
        Pl = legendre(l)(x_vals)
        Pl_minus_1 = legendre(l-1)(x_vals)
        Pl_minus_2 = legendre(l-2)(x_vals)
        
        data_node["l_order"].extend([[0.0, float(l)] for _ in range(num_samples_per_l)])
        data_node["x"].extend([[0.0, float(xv)] for xv in x_vals])
        data_node["p_prev1"].extend([[0.0, float(p1)] for p1 in Pl_minus_1])
        data_node["p_prev2"].extend([[0.0, float(p2)] for p2 in Pl_minus_2])
        targets.extend([[0.0, float(pl)] for pl in Pl])
    
    dataset = {"V": 2, "E": 1, "A": [[0,1],[0,0]], "G": [[0,1]],
               "l_order": data_node["l_order"],
               "x": data_node["x"],
               "p_prev1": data_node["p_prev1"],
               "p_prev2": data_node["p_prev2"],
               "target": targets}
    
    os.makedirs("Multipolos/data/nd2_format", exist_ok=True)
    json.dump(dataset, open("Multipolos/data/nd2_format/RECURRENCE_nd2.json", "w"))
    print(f"Dataset de Recurrencia (l={all_l}) generado.")

generate_recurrence_data()

In [ ]:
# 3. BÚSQUEDA DE LA LEY RECURSIVA (Fórmula de Bonnet)
!python3 search.py \
    --data Multipolos/data/nd2_format/RECURRENCE_nd2.json \
    --vars l_order x p_prev1 p_prev2 \
    --target_var target \
    --episodes 3000 \
    --beam_size 10 \
    --device cuda

In [ ]:
# 4. AUTO-PUSH
def get_token_reloaded():
    try: from kaggle_secrets import UserSecretsClient; return UserSecretsClient().get_secret("GITHUB_TOKEN")
    except: pass
    try: from google.colab import userdata; return userdata.get('GITHUB_TOKEN')
    except: pass
    return None
token = get_token_reloaded()
if token:
    os.system("git config --global user.email 'manuel@researcher.phys'")
    os.system("git config --global user.name 'ND2-Auto-Agent'")
    os.system(f"git remote set-url origin https://{token}@github.com/manramirezpi/ND2.git")
    !git add Multipolos/results/ Multipolos/data/nd2_format/RECURRENCE_nd2.json
    !git commit -m "Fase 6.5: Descubrimiento de Recurrencia de Legendre (Bonnet Formula)"
    !git push origin main
else: print("\n[!] GITHUB_TOKEN no encontrado.")